In [5]:
from __future__ import annotations

import json
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Any

from agents import ModelSettings, OpenAIResponsesModel, Runner
from agents.run import RunConfig
from agents.sandbox import FileMode, Manifest, Permissions, SandboxAgent, SandboxRunConfig, SandboxPathGrant, User
from agents.sandbox.capabilities import Filesystem, LocalDirLazySkillSource, Shell, Skills
from agents.sandbox.entries import Dir, File, LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient
from openai import AsyncOpenAI


# %% [markdown]
# ## 1. 基础路径和模型配置

# %%
def resolve_paths(start: Path) -> tuple[Path, Path]:
    start = start.resolve()
    for path in (start, *start.parents):
        if path.name == "02-sandbox" and path.parent.name == "notebooks":
            if (path / "repo").is_dir() and (path / "skills").is_dir():
                return path.parents[1], path
        candidate = path / "notebooks" / "02-sandbox"
        if (candidate / "repo").is_dir() and (candidate / "skills").is_dir():
            return path, candidate
    raise RuntimeError(f"Cannot resolve notebook sandbox paths from {start}")


PROJECT_ROOT, EXAMPLE_DIR = resolve_paths(Path.cwd())
HOST_REPO_DIR = EXAMPLE_DIR / "repo"
HOST_SKILLS_DIR = EXAMPLE_DIR / "skills"
LOG_DIR = PROJECT_ROOT / "logs"

MODEL = "gpt-5.3-codex"
BASE_URL = "https://api.tokenlab.sh/v1"
API_KEY = "sk-aeRemEo2sD0YgQWEFGjipWrzTp4LVFUVzHD8bD5fx5PoLMGF"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("EXAMPLE_DIR  =", EXAMPLE_DIR)
print("HOST_REPO_DIR=", HOST_REPO_DIR)
print("HOST_SKILLS_DIR=", HOST_SKILLS_DIR)
print("MODEL=", MODEL)
print("BASE_URL=", BASE_URL)
print("API_KEY set=", bool(API_KEY))

if not API_KEY:
    raise RuntimeError("请先设置 OPENAI_API_KEY 环境变量")


PROJECT_ROOT = /Users/zhangtianzhu/Project/huya/ANIFORCE
EXAMPLE_DIR  = /Users/zhangtianzhu/Project/huya/ANIFORCE/notebooks/02-sandbox
HOST_REPO_DIR= /Users/zhangtianzhu/Project/huya/ANIFORCE/notebooks/02-sandbox/repo
HOST_SKILLS_DIR= /Users/zhangtianzhu/Project/huya/ANIFORCE/notebooks/02-sandbox/skills
MODEL= gpt-5.3-codex
BASE_URL= https://api.tokenlab.sh/v1
API_KEY set= True


In [6]:

# %% [markdown]
# ## 2. 小工具：JSON dump 和日志

# %%
def now_label() -> str:
    return datetime.now().strftime("%y%m%d_%H%M%S")


def dump(obj: Any) -> Any:
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    try:
        return json.loads(json.dumps(obj, default=str))
    except TypeError:
        return str(obj)


def pretty(obj: Any, limit: int = 5000) -> None:
    text = json.dumps(dump(obj), ensure_ascii=False, indent=2, default=str)
    print(text[:limit])
    if len(text) > limit:
        print(f"\n... truncated, total chars={len(text)}")


class JsonlLogger:
    def __init__(self, path: Path) -> None:
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def write(self, event: str, payload: dict[str, Any] | None = None) -> None:
        record = {
            "ts": datetime.now().isoformat(timespec="milliseconds"),
            "event": event,
            "payload": payload or {},
        }
        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


log_path = LOG_DIR / f"sandbox_core_cells_{now_label()}.jsonl"
logger = JsonlLogger(log_path)
print("log_path=", log_path)

log_path= /Users/zhangtianzhu/Project/huya/ANIFORCE/logs/sandbox_core_cells_260701_111603.jsonl


In [7]:
client = AsyncOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    timeout=90.0,
    max_retries=0,
)

model = OpenAIResponsesModel(
    model=MODEL,
    openai_client=client,
)

logger.write("model", {"model": MODEL, "base_url": BASE_URL})
print(model)


In [8]:
# %% [markdown]
# ## 4. Manifest 最小清单：File / Dir / LocalDir
#
# 这个 cell 只构造清单，不调用模型。
#
# 你要观察三件事：
# - entries 的 key 是沙盒工作区内路径，例如 repo、workspace_notes/task.md、output。
# - LocalDir(src=HOST_REPO_DIR) 是把宿主机目录物化到沙盒里的 repo/。
# - Dir() 只是在沙盒里创建目录，它不会读取宿主机同名目录。

# %%
def assert_manifest_entry_paths(manifest: Manifest) -> None:
    """Notebook-side guard: catch non-portable workspace entry paths early."""
    bad_paths: list[str] = []
    for raw_path in manifest.entries:
        path = Path(raw_path)
        if path.is_absolute() or ".." in path.parts:
            bad_paths.append(str(raw_path))
    if bad_paths:
        raise ValueError(f"Manifest entry paths must be relative and stay inside workspace: {bad_paths}")


private_permissions = Permissions(
    owner=FileMode.READ | FileMode.WRITE,
    group=FileMode.NONE,
    other=FileMode.NONE,
)
output_permissions = Permissions(
    owner=FileMode.ALL,
    group=FileMode.ALL,
    other=FileMode.NONE,
)

basic_manifest = Manifest(
    root=str(EXAMPLE_DIR),
    users=[User(name="analyst")],
    environment={
        "DEMO_ENV": "manifest-env-visible-in-sandbox",
    },
    entries={
        "repo": LocalDir(src=HOST_REPO_DIR),
        "workspace_notes/task.md": File(
            content=(
                b"# Manifest demo task\n\n"
                b"1. Read repo/task.md.\n"
                b"2. Write a short report to output/manifest_report.md.\n"
            ),
            permissions=private_permissions,
        ),
        "output": Dir(permissions=output_permissions),
    },
)

assert_manifest_entry_paths(basic_manifest)
pretty({
    "root": basic_manifest.root,
    "entries": {
        str(path): type(entry).__name__
        for path, entry in basic_manifest.entries.items()
    },
    "environment": dump(basic_manifest.environment),
    "users": [user.name for user in basic_manifest.users],
    "host_sources": {
        "HOST_REPO_DIR": str(HOST_REPO_DIR),
        "HOST_SKILLS_DIR": str(HOST_SKILLS_DIR),
    },
})


{
  "root": "/Users/zhangtianzhu/Project/huya/ANIFORCE/notebooks/02-sandbox",
  "entries": {
    "repo": "LocalDir",
    "workspace_notes/task.md": "File",
    "output": "Dir"
  },
  "environment": {
    "value": {}
  },
  "users": [
    "analyst"
  ],
  "host_sources": {
    "HOST_REPO_DIR": "/Users/zhangtianzhu/Project/huya/ANIFORCE/notebooks/02-sandbox/repo",
    "HOST_SKILLS_DIR": "/Users/zhangtianzhu/Project/huya/ANIFORCE/notebooks/02-sandbox/skills"
  }
}


In [ ]:
from agents.sandbox.manifest import Environment                                                
                                                                                                  
basic_manifest = Manifest(                                                                     
    root="/workspace",                                                                         
    users=[User(name="analyst")],                                                              
    environment=Environment(value={                                                            
        "DEMO_ENV": "manifest-env-visible-in-sandbox",                                         
    }),                                                                                        
    entries={                                                                                  
        "repo": LocalDir(src=HOST_REPO_DIR),                                                   
        "workspace_notes/task.md": File(                                                       
            content=(                                                                          
                b"# Manifest demo task\n\n"                                                    
                b"1. Read repo/task.md.\n"                                                     
                b"2. Write a short report to output/manifest_report.md.\n"                     
            ),                                                                                 
            permissions=private_permissions,                                                   
        ),                                                                                     
        "output": Dir(permissions=output_permissions),                                         
    },                                                                                         
)            